# Phase 2: Semi-Automated VLM Error Annotation\n\nUse LLaVA (4-bit quantized) to annotate image editing errors across 11 error categories.\n\n**Input:** `phase1_dataset.parquet` + images\n**Output:** `phase2_annotated.parquet`

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers bitsandbytes accelerate Pillow pandas torch

In [ ]:
# Cell 2: Config & Utils
import os
import sys
import json
import re
import numpy as np
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

# ── Path Detection ──
IS_KAGGLE = os.path.exists("/kaggle/working")
OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "./output"
INPUT_DIR = "/kaggle/input" if IS_KAGGLE else "./output"

# Phase 1 output location (on Kaggle, add as dataset named "phase1-output")
PHASE1_DIR = os.path.join(INPUT_DIR, "phase1-output") if IS_KAGGLE else OUTPUT_DIR

# ── Error Taxonomy ──
ERROR_TAXONOMY = {
    0: "Wrong Object", 1: "Missing Object", 2: "Extra Object",
    3: "Wrong Attribute", 4: "Spatial Error", 5: "Style Mismatch",
    6: "Over-editing", 7: "Under-editing", 8: "Artifact/Quality",
    9: "Ambiguous Prompt", 10: "Failed Removal",
}
NUM_ERROR_CLASSES = 11

# ── Annotation ──
ANNOTATION_MODEL = "llava-hf/llava-v1.6-mistral-7b-hf"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"IS_KAGGLE: {IS_KAGGLE}")
print(f"PHASE1_DIR: {PHASE1_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


def concat_images_side_by_side(img1, img2, target_height=384):
    """Concatenate two images side by side for VLM input."""
    if isinstance(img1, str):
        img1 = Image.open(img1).convert("RGB")
    if isinstance(img2, str):
        img2 = Image.open(img2).convert("RGB")
    ratio1 = target_height / img1.height
    ratio2 = target_height / img2.height
    img1 = img1.resize((int(img1.width * ratio1), target_height), Image.LANCZOS)
    img2 = img2.resize((int(img2.width * ratio2), target_height), Image.LANCZOS)
    total_width = img1.width + img2.width
    combined = Image.new("RGB", (total_width, target_height))
    combined.paste(img1, (0, 0))
    combined.paste(img2, (img1.width, 0))
    return combined


def parse_error_labels_from_text(text):
    """Parse VLM output to extract error labels."""
    result = {
        "error_labels": [0] * NUM_ERROR_CLASSES,
        "error_types": [],
        "has_error": False,
        "confidence": 0.0,
    }
    # Try JSON parsing
    try:
        json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            if "error_types" in parsed:
                error_types = parsed["error_types"]
                if isinstance(error_types, list):
                    taxonomy_lower = {v.lower(): k for k, v in ERROR_TAXONOMY.items()}
                    for et in error_types:
                        et_lower = et.strip().lower()
                        if et_lower in taxonomy_lower:
                            idx = taxonomy_lower[et_lower]
                            result["error_labels"][idx] = 1
                            result["error_types"].append(ERROR_TAXONOMY[idx])
            if "has_error" in parsed:
                result["has_error"] = bool(parsed["has_error"])
            if "confidence" in parsed:
                result["confidence"] = float(parsed["confidence"])
            if any(result["error_labels"]):
                result["has_error"] = True
            return result
    except (json.JSONDecodeError, ValueError, KeyError):
        pass
    # Fallback: regex
    text_lower = text.lower()
    taxonomy_lower = {v.lower(): k for k, v in ERROR_TAXONOMY.items()}
    for name_lower, idx in taxonomy_lower.items():
        if name_lower in text_lower:
            result["error_labels"][idx] = 1
            result["error_types"].append(ERROR_TAXONOMY[idx])
            result["has_error"] = True
    if result["has_error"]:
        result["confidence"] = 0.5
    return result

In [ ]:
# Cell 3: Load Phase 1 data
parquet_path = os.path.join(PHASE1_DIR, "phase1_dataset.parquet")
df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df)} samples from Phase 1")
print(f"Columns: {list(df.columns)}")
print(df.head(3))

In [ ]:
# Cell 4: Annotation prompt template
ANNOTATION_PROMPT_TEMPLATE = """You are an expert image editing quality annotator. You are given a side-by-side image showing the ORIGINAL image (left) and the EDITED image (right). The edit instruction was: "{edit_prompt}"

Analyze the editing result and identify ALL errors from this taxonomy:
0: Wrong Object - incorrect object was modified or replaced
1: Missing Object - an object that should be present is missing
2: Extra Object - unwanted objects appeared in the edit
3: Wrong Attribute - color, texture, size, or other attribute is wrong
4: Spatial Error - object is in the wrong position or has wrong orientation
5: Style Mismatch - editing style doesn't match the instruction
6: Over-editing - too many changes beyond what was requested
7: Under-editing - the edit is incomplete or insufficient
8: Artifact/Quality - visual artifacts, blurriness, or quality degradation
9: Ambiguous Prompt - the instruction is unclear or ambiguous
10: Failed Removal - object removal was attempted but failed

Respond ONLY with a JSON object in this exact format:
{{"has_error": true/false, "error_types": ["Error Name 1", "Error Name 2"], "confidence": 0.0-1.0}}

If the edit is perfect with no errors, respond:
{{"has_error": false, "error_types": [], "confidence": 0.95}}
"""

print("Annotation prompt template defined.")
print(f"Template length: {len(ANNOTATION_PROMPT_TEMPLATE)} chars")

In [ ]:
# Cell 5: Load LLaVA model (4-bit quantized for T4)
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {ANNOTATION_MODEL} with 4-bit quantization...")
processor = LlavaNextProcessor.from_pretrained(ANNOTATION_MODEL)
model = LlavaNextForConditionalGeneration.from_pretrained(
    ANNOTATION_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Model loaded successfully.")

In [ ]:
# Cell 6: Annotation function
@torch.no_grad()
def annotate_single(orig_img_path, edit_img_path, edit_prompt, max_retries=3):
    """Annotate a single sample using LLaVA."""
    # Build side-by-side image
    orig_full = os.path.join(PHASE1_DIR, orig_img_path)
    edit_full = os.path.join(PHASE1_DIR, edit_img_path)
    combined_img = concat_images_side_by_side(orig_full, edit_full)
    
    prompt_text = ANNOTATION_PROMPT_TEMPLATE.format(edit_prompt=edit_prompt)
    
    # LLaVA conversation format
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text},
            ],
        },
    ]
    
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=combined_img, text=prompt, return_tensors="pt").to(model.device)
    
    for attempt in range(max_retries):
        try:
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=1.0,
            )
            # Decode only new tokens
            generated = processor.decode(output[0][inputs["input_ids"].shape[1]:], 
                                         skip_special_tokens=True)
            result = parse_error_labels_from_text(generated)
            
            if result["confidence"] > 0 or not result["has_error"]:
                result["raw_output"] = generated
                return result
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
    
    # Return default on all failures
    return {
        "error_labels": [0] * NUM_ERROR_CLASSES,
        "error_types": [],
        "has_error": False,
        "confidence": 0.0,
        "raw_output": "FAILED",
    }

print("Annotation function defined.")

In [ ]:
# Cell 7: Run annotation loop with checkpointing
CHECKPOINT_EVERY = 100
checkpoint_path = os.path.join(OUTPUT_DIR, "phase2_checkpoint.parquet")

# Initialize annotation columns
if "error_labels" not in df.columns:
    df["error_labels"] = [None] * len(df)
    df["error_types"] = [None] * len(df)
    df["has_error"] = [None] * len(df)
    df["annotation_confidence"] = [None] * len(df)
    df["raw_vlm_output"] = [None] * len(df)

# Check for existing checkpoint
start_idx = 0
if os.path.exists(checkpoint_path):
    df_ckpt = pd.read_parquet(checkpoint_path)
    # Find where we left off
    annotated_mask = df_ckpt["annotation_confidence"].notna()
    if annotated_mask.any():
        start_idx = annotated_mask.sum()
        df = df_ckpt
        print(f"Resuming from checkpoint at index {start_idx}")

print(f"Annotating {len(df) - start_idx} remaining samples...")

for idx in tqdm(range(start_idx, len(df)), initial=start_idx, total=len(df)):
    row = df.iloc[idx]
    
    result = annotate_single(
        row["original_image_path"],
        row["edited_image_path"],
        row["edit_prompt"],
    )
    
    df.at[idx, "error_labels"] = json.dumps(result["error_labels"])
    df.at[idx, "error_types"] = json.dumps(result["error_types"])
    df.at[idx, "has_error"] = result["has_error"]
    df.at[idx, "annotation_confidence"] = result["confidence"]
    df.at[idx, "raw_vlm_output"] = result.get("raw_output", "")
    
    # Periodic checkpoint
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        df.to_parquet(checkpoint_path, index=False)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  Checkpoint saved at {idx + 1}/{len(df)}")

print("Annotation complete!")

In [ ]:
# Cell 8: Validation
# Parse error_labels from JSON strings to lists
df["error_labels_parsed"] = df["error_labels"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Validate all vectors are length 11
invalid_count = sum(1 for labels in df["error_labels_parsed"] if labels is None or len(labels) != NUM_ERROR_CLASSES)
print(f"Invalid label vectors: {invalid_count}/{len(df)}")

# Error distribution
error_counts = np.zeros(NUM_ERROR_CLASSES)
for labels in df["error_labels_parsed"]:
    if labels is not None and len(labels) == NUM_ERROR_CLASSES:
        error_counts += np.array(labels)

print(f"\nError type distribution:")
for i, count in enumerate(error_counts):
    print(f"  {ERROR_TAXONOMY[i]}: {int(count)} ({count/len(df)*100:.1f}%)")

# Confidence stats
conf = df["annotation_confidence"].astype(float)
print(f"\nConfidence stats:")
print(f"  Mean: {conf.mean():.3f}")
print(f"  Median: {conf.median():.3f}")
print(f"  Low confidence (<0.5): {(conf < 0.5).sum()}/{len(df)}")

# Has error stats
has_err = df["has_error"].astype(bool)
print(f"\nSamples with errors: {has_err.sum()}/{len(df)} ({has_err.sum()/len(df)*100:.1f}%)")
print(f"Samples without errors: {(~has_err).sum()}/{len(df)}")

In [ ]:
# Cell 9: Save annotated dataset
# Drop parsed column (keep JSON strings for parquet compatibility)
df_save = df.drop(columns=["error_labels_parsed"], errors="ignore")
output_path = os.path.join(OUTPUT_DIR, "phase2_annotated.parquet")
df_save.to_parquet(output_path, index=False)
print(f"Saved annotated dataset to {output_path}")
print(f"Shape: {df_save.shape}")

# Cleanup checkpoint
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Removed checkpoint file.")

In [ ]:
# Cell 10: Visualize error distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Error type bar chart
error_names = [ERROR_TAXONOMY[i] for i in range(NUM_ERROR_CLASSES)]
axes[0].barh(error_names, error_counts, color="steelblue")
axes[0].set_xlabel("Count")
axes[0].set_title("Error Type Distribution")
axes[0].invert_yaxis()

# Confidence histogram
axes[1].hist(conf.dropna(), bins=50, color="coral", edgecolor="black", alpha=0.7)
axes[1].axvline(x=0.5, color="red", linestyle="--", label="Low conf threshold")
axes[1].set_xlabel("Confidence")
axes[1].set_ylabel("Count")
axes[1].set_title("Annotation Confidence Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "phase2_visualization.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved.")